# Unsteady Cylinder Flow: Vortex Shedding Visualization

This notebook visualizes **time-dependent** flow around a cylinder, capturing the famous **von Kármán vortex street**.

**Requirements**: FEniCSx is required for data generation. Demo data is provided if unavailable.

```bash
conda install -c conda-forge fenics-dolfinx mpich pyvista
```

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Circle
from matplotlib.colors import Normalize
from mpl_toolkits.axes_grid1 import make_axes_locatable

import sys
sys.path.insert(0, '..')

# Check for widgets
try:
    import ipywidgets as widgets
    from ipywidgets import interact, interactive, IntSlider, FloatSlider, Dropdown, Checkbox, Play
    from IPython.display import display, clear_output
    HAS_WIDGETS = True
except ImportError:
    HAS_WIDGETS = False
    print("ipywidgets not found. Install with: pip install ipywidgets")

# Check for FEniCSx
try:
    from pdeforge import generate_dataset, list_models, describe_model
    if 'cylinder_flow_2d_unsteady' in list_models():
        HAS_FENICSX = True
        print("FEniCSx available. Unsteady cylinder flow model ready.")
    else:
        HAS_FENICSX = False
        print("FEniCSx not available. Using synthetic demo data.")
except Exception as e:
    HAS_FENICSX = False
    print(f"Import error: {e}")

FEniCSx available. Unsteady cylinder flow model ready.


## 1. Configuration and Geometry

In [2]:
# Geometry configuration
CYLINDER_CENTER = (0.2, 0.2)
CYLINDER_RADIUS = 0.05
CHANNEL_LENGTH = 2.2
CHANNEL_HEIGHT = 0.41

# Resolution
NX, NY = 110, 41
RESOLUTION = {"x": NX, "y": NY}

# Time settings
TIME_END = 8.0  # seconds
N_TIME_STEPS = 41  # Number of output frames (reduced for demo)

print(f"Domain: {CHANNEL_LENGTH} x {CHANNEL_HEIGHT} m")
print(f"Cylinder: center={CYLINDER_CENTER}, radius={CYLINDER_RADIUS}")
print(f"Resolution: {NX} x {NY} grid points")
print(f"Time: 0 to {TIME_END} s in {N_TIME_STEPS} steps")

Domain: 2.2 x 0.41 m
Cylinder: center=(0.2, 0.2), radius=0.05
Resolution: 110 x 41 grid points
Time: 0 to 8.0 s in 41 steps


## 2. Generate or Load Data

In [3]:
if HAS_FENICSX:
    print(describe_model("cylinder_flow_2d_unsteady"))

Model: cylinder_flow_2d_unsteady
Dimensions: 2D


        2D unsteady flow around a cylinder using FEniCSx.
        
        Solves time-dependent Navier-Stokes equations to capture vortex
        shedding dynamics behind a circular cylinder.
        
        Parameters
        ----------
        resolution : Dict[str, int]
            Output grid resolution, e.g., {"x": 128, "y": 64}
        inlet_velocity : float
            Mean inlet velocity (default: 1.0)
        viscosity : float
            Dynamic viscosity μ (default: 0.001)
        time_end : float
            Final simulation time (default: 8.0)
        n_time_steps : int
            Number of output time steps (default: 81)
        
        Examples
        --------
        >>> dataset = generate_dataset(
        ...     model="cylinder_flow_2d_unsteady",
        ...     n_samples=5,
        ...     resolution={"x": 110, "y": 41},
        ...     params={"inlet_velocity": 1.0, "time_end": 8.0},
        ... )
        

Inpu

In [4]:
if HAS_FENICSX:
    from pdeforge import get_model
    
    print("Generating unsteady cylinder flow trajectory...")
    print("(This may take several minutes for the time-stepping simulation)")
    
    # Create model
    Model = get_model("cylinder_flow_2d_unsteady")
    model = Model(
        resolution=RESOLUTION,
        inlet_velocity=1.0,
        viscosity=0.001,
        time_end=TIME_END,
        _n_time_steps=N_TIME_STEPS,
    )
    
    # Generate a single trajectory
    inlet_scale = np.array([1.0])
    trajectory = model.solve(inlet_scale=1.0, return_full=True)
    
    # Grid
    x = model.grids['x']
    y = model.grids['y']
    t = np.linspace(0, TIME_END, N_TIME_STEPS)
    
    print(f"\nTrajectory shape: {trajectory.shape}")
    print(f"  (n_time={trajectory.shape[0]}, ny={trajectory.shape[1]}, nx={trajectory.shape[2]}, channels={trajectory.shape[3]})")

else:
    # Create synthetic demo data that mimics vortex shedding
    print("Creating synthetic vortex shedding demo data...")
    
    x = np.linspace(0, CHANNEL_LENGTH, NX)
    y = np.linspace(0, CHANNEL_HEIGHT, NY)
    t = np.linspace(0, TIME_END, N_TIME_STEPS)
    X, Y = np.meshgrid(x, y)
    
    def create_demo_trajectory():
        """Create synthetic vortex shedding data."""
        trajectory = []
        
        # Strouhal number for vortex shedding frequency
        St = 0.2
        U_mean = 1.0
        D = 2 * CYLINDER_RADIUS
        f_shedding = St * U_mean / D  # Shedding frequency
        
        for i, ti in enumerate(t):
            # Base parabolic flow
            U_max = 1.5 * U_mean
            u_base = 4 * U_max * Y * (CHANNEL_HEIGHT - Y) / CHANNEL_HEIGHT**2
            v_base = np.zeros_like(u_base)
            
            # Wake region
            wake_x = X - CYLINDER_CENTER[0]
            wake_y = Y - CYLINDER_CENTER[1]
            
            # Oscillating wake (von Kármán street)
            phase = 2 * np.pi * f_shedding * ti
            wake_amplitude = 0.3 * np.sin(phase)
            
            # Wake velocity deficit and oscillation
            wake_decay = np.exp(-wake_x / 0.5) * (wake_x > 0).astype(float)
            wake_width = 0.05 + 0.02 * wake_x * (wake_x > 0).astype(float)
            
            # Alternating vortices
            vortex_y_offset = wake_amplitude * np.sin(5 * wake_x - phase)
            wake_profile = np.exp(-(wake_y - vortex_y_offset)**2 / (2 * wake_width**2))
            
            u = u_base * (1 - 0.6 * wake_decay * wake_profile)
            v = 0.3 * wake_decay * wake_profile * np.cos(5 * wake_x - phase)
            
            # Zero inside cylinder
            r = np.sqrt(wake_x**2 + wake_y**2)
            inside = r < CYLINDER_RADIUS
            u[inside] = 0
            v[inside] = 0
            
            # Pressure (simplified)
            p = -0.5 * (X - 1.1) + 0.1 * np.sin(5 * wake_x - phase) * wake_decay
            p[inside] = 0
            
            trajectory.append(np.stack([u, v, p], axis=-1))
        
        return np.stack(trajectory, axis=0)
    
    trajectory = create_demo_trajectory()
    inlet_scale = np.array([1.0])
    
    print(f"Demo trajectory shape: {trajectory.shape}")

Generating unsteady cylinder flow trajectory...
(This may take several minutes for the time-stepping simulation)
Info    : Meshing 1D...                                                                                                                        
Info    : [  0%] Meshing curve 5 (Ellipse)
Info    : [ 30%] Meshing curve 6 (Line)
Info    : [ 50%] Meshing curve 7 (Line)
Info    : [ 70%] Meshing curve 8 (Line)
Info    : [ 90%] Meshing curve 9 (Line)
Info    : Done meshing 1D (Wall 0.00428542s, CPU 0.007418s)
Info    : Meshing 2D...
Info    : Meshing surface 1 (Plane, Frontal-Delaunay)
Info    : Done meshing 2D (Wall 0.072567s, CPU 0.127549s)
Info    : 4015 nodes 8035 elements


ld: warning: duplicate -rpath '/Users/pyatsyshin/miniconda3/envs/pdeforge-fenicsx/lib' ignored
ld: warning: duplicate -rpath '/Users/pyatsyshin/miniconda3/envs/pdeforge-fenicsx/lib' ignored
ld: warning: duplicate -rpath '/Users/pyatsyshin/miniconda3/envs/pdeforge-fenicsx/lib' ignored
ld: warning: duplicate -rpath '/Users/pyatsyshin/miniconda3/envs/pdeforge-fenicsx/lib' ignored



Trajectory shape: (41, 110, 41, 3)
  (n_time=41, ny=110, nx=41, channels=3)


## 3. Visualization Utilities

In [5]:
def add_cylinder(ax, center=CYLINDER_CENTER, radius=CYLINDER_RADIUS):
    """Add cylinder patch to axes."""
    circle = Circle(center, radius, color='gray', ec='black', lw=2, zorder=10)
    ax.add_patch(circle)


def plot_frame(ax, u, v, p, x, y, t_val, field='velocity', show_streamlines=True, cmap='viridis'):
    """Plot a single frame of the flow."""
    ax.clear()
    
    if field == 'velocity':
        vmag = np.sqrt(u**2 + v**2)
        vmax = vmag.max()
        im = ax.contourf(x, y, vmag, levels=30, cmap=cmap, vmin=0, vmax=vmax)
        if show_streamlines:
            ax.streamplot(x, y, u, v, color='white', density=1.2, linewidth=0.6, arrowsize=0.6)
        label = '|u| (m/s)'
        
    elif field == 'vorticity':
        dx = x[1] - x[0]
        dy = y[1] - y[0]
        dvdx = np.gradient(v, dx, axis=1)
        dudy = np.gradient(u, dy, axis=0)
        omega = dvdx - dudy
        vmax = np.abs(omega).max()
        im = ax.contourf(x, y, omega, levels=30, cmap='RdBu_r', vmin=-vmax, vmax=vmax)
        label = 'ω (1/s)'
        
    elif field == 'pressure':
        vmax = np.abs(p).max()
        if vmax < 1e-10: vmax = 1.0
        im = ax.contourf(x, y, p, levels=30, cmap='RdBu_r', vmin=-vmax, vmax=vmax)
        label = 'p (Pa)'
        
    elif field == 'v_velocity':
        vmax = np.abs(v).max()
        if vmax < 1e-10: vmax = 0.1
        im = ax.contourf(x, y, v, levels=30, cmap='RdBu_r', vmin=-vmax, vmax=vmax)
        label = 'v (m/s)'
    
    add_cylinder(ax)
    ax.set_xlabel('x (m)')
    ax.set_ylabel('y (m)')
    ax.set_title(f't = {t_val:.3f} s')
    ax.set_aspect('equal')
    ax.set_xlim(0, CHANNEL_LENGTH)
    ax.set_ylim(0, CHANNEL_HEIGHT)
    
    return im, label

## 4. Time Snapshots

View snapshots of the flow at different times.

In [ ]:
# Select time snapshots
n_snapshots = 6
snapshot_indices = np.linspace(0, len(t) - 1, n_snapshots, dtype=int)

fig, axes = plt.subplots(2, 3, figsize=(16, 8))
axes = axes.flatten()

for i, t_idx in enumerate(snapshot_indices):
    u = trajectory[t_idx, :, :, 0].T
    v = trajectory[t_idx, :, :, 1].T
    p = trajectory[t_idx, :, :, 2].T
    
    im, label = plot_frame(axes[i], u, v, p, x, y, t[t_idx], field='velocity')

fig.suptitle('Flow Evolution: Velocity Magnitude', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Vorticity snapshots (shows vortex shedding more clearly)
fig, axes = plt.subplots(2, 3, figsize=(16, 8))
axes = axes.flatten()

for i, t_idx in enumerate(snapshot_indices):
    u = trajectory[t_idx, :, :, 0].T
    v = trajectory[t_idx, :, :, 1].T
    p = trajectory[t_idx, :, :, 2].T
    
    im, label = plot_frame(axes[i], u, v, p, x, y, t[t_idx], field='vorticity', show_streamlines=False)

fig.suptitle('Vorticity Field: Von Kármán Vortex Street', fontsize=14)
plt.tight_layout()
plt.show()

## 5. Interactive Time Slider

Use the slider to scrub through time and watch vortices develop.

In [ ]:
if HAS_WIDGETS:
    def explore_time(time_idx, field='velocity', show_streamlines=True):
        """Interactive time exploration."""
        u = trajectory[time_idx, :, :, 0].T
        v = trajectory[time_idx, :, :, 1].T
        p = trajectory[time_idx, :, :, 2].T
        t_val = t[time_idx]
        
        fig, ax = plt.subplots(1, 1, figsize=(14, 5))
        im, label = plot_frame(ax, u, v, p, x, y, t_val, field=field, show_streamlines=show_streamlines)
        plt.colorbar(im, ax=ax, label=label, shrink=0.8)
        plt.tight_layout()
        plt.show()
        
        # Statistics
        vmag = np.sqrt(u**2 + v**2)
        print(f"Time: {t_val:.3f} / {TIME_END:.1f} s")
        print(f"Max velocity: {vmag.max():.4f} m/s")
        print(f"v-velocity range: [{v.min():.4f}, {v.max():.4f}] m/s")
    
    time_slider = IntSlider(
        min=0, max=len(t) - 1, step=1, value=len(t) // 2,
        description='Time step:', continuous_update=False
    )
    field_dropdown = Dropdown(
        options=['velocity', 'vorticity', 'pressure', 'v_velocity'],
        value='velocity', description='Field:'
    )
    streamlines_check = Checkbox(value=True, description='Streamlines')
    
    interactive_time = interactive(
        explore_time,
        time_idx=time_slider,
        field=field_dropdown,
        show_streamlines=streamlines_check,
    )
    display(interactive_time)
else:
    print("Interactive widgets not available.")

## 6. Animation with Play Button

Click Play to animate the vortex shedding.

In [ ]:
if HAS_WIDGETS:
    # Play widget for animation
    play = Play(
        value=0,
        min=0,
        max=len(t) - 1,
        step=1,
        interval=100,  # milliseconds per frame
        description="Play",
    )
    
    anim_slider = IntSlider(
        min=0, max=len(t) - 1, step=1, value=0,
        description='Frame:',
        continuous_update=True,
        readout=False,
    )
    
    widgets.jslink((play, 'value'), (anim_slider, 'value'))
    
    field_select = Dropdown(
        options=['velocity', 'vorticity', 'pressure'],
        value='vorticity', description='Field:'
    )
    
    output = widgets.Output()
    
    def update_animation(change):
        with output:
            clear_output(wait=True)
            
            t_idx = anim_slider.value
            field = field_select.value
            
            u = trajectory[t_idx, :, :, 0].T
            v = trajectory[t_idx, :, :, 1].T
            p = trajectory[t_idx, :, :, 2].T
            t_val = t[t_idx]
            
            fig, ax = plt.subplots(1, 1, figsize=(14, 5))
            im, label = plot_frame(ax, u, v, p, x, y, t_val, field=field, 
                                   show_streamlines=(field == 'velocity'))
            plt.colorbar(im, ax=ax, label=label, shrink=0.8)
            
            # Progress bar
            progress = t_val / TIME_END
            ax.text(0.02, 0.02, f'Progress: {progress*100:.0f}%', transform=ax.transAxes,
                   fontsize=10, bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))
            
            plt.tight_layout()
            plt.show()
    
    anim_slider.observe(update_animation, names='value')
    field_select.observe(update_animation, names='value')
    
    controls = widgets.HBox([play, anim_slider, field_select])
    display(widgets.VBox([controls, output]))
    
    update_animation(None)
else:
    print("Animation requires ipywidgets.")

## 7. Space-Time Diagram

Visualize the oscillation at a fixed y-position (centerline).

In [ ]:
# Extract v-velocity along the centerline (y = center of cylinder)
y_idx = np.argmin(np.abs(y - CYLINDER_CENTER[1]))

# v-velocity at this y-position over time
# trajectory shape is (n_time, nx, ny, channels), so index y in the 3rd dimension
v_centerline = trajectory[:, :, y_idx, 1]  # Shape: (n_time, nx)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Space-time diagram
vmax = np.abs(v_centerline).max()
if vmax < 1e-10: vmax = 0.1

im1 = axes[0].imshow(
    v_centerline,
    aspect='auto',
    origin='lower',
    extent=[x[0], x[-1], t[0], t[-1]],
    cmap='RdBu_r',
    vmin=-vmax, vmax=vmax
)
axes[0].axvline(CYLINDER_CENTER[0], color='gray', linestyle='--', linewidth=2, label='Cylinder')
axes[0].set_xlabel('x (m)')
axes[0].set_ylabel('Time (s)')
axes[0].set_title(f'v-velocity at y = {y[y_idx]:.3f} m (Space-Time)')
axes[0].legend(loc='upper left')
plt.colorbar(im1, ax=axes[0], label='v (m/s)')

# Time series at a probe location (behind cylinder)
x_probe = 0.5  # Behind cylinder
x_idx = np.argmin(np.abs(x - x_probe))

axes[1].plot(t, v_centerline[:, x_idx], 'b-', linewidth=1.5)
axes[1].axhline(0, color='gray', linestyle='--', alpha=0.5)
axes[1].set_xlabel('Time (s)')
axes[1].set_ylabel('v-velocity (m/s)')
axes[1].set_title(f'v-velocity Time Series at x={x_probe}, y={y[y_idx]:.3f}')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("The oscillation in v-velocity indicates vortex shedding.")
print(f"Probe location: x = {x_probe} m (behind cylinder at x = {CYLINDER_CENTER[0]} m)")

## 8. Lift and Drag Forces (Approximate)

Compute approximate lift and drag coefficients from the flow.

In [ ]:
# Approximate forces by integrating pressure around a contour near the cylinder
# This is a simplified estimate using the pressure field

# Extract pressure along a circle around the cylinder
n_angles = 64
theta = np.linspace(0, 2*np.pi, n_angles)
r_sample = CYLINDER_RADIUS * 1.3  # Slightly outside cylinder

# Sample points
x_circle = CYLINDER_CENTER[0] + r_sample * np.cos(theta)
y_circle = CYLINDER_CENTER[1] + r_sample * np.sin(theta)

# Interpolate pressure at these points over time
from scipy.interpolate import RegularGridInterpolator

lift_forces = []
drag_forces = []

for t_idx in range(len(t)):
    p = trajectory[t_idx, :, :, 2].T  # Transpose to shape (ny, nx) for interpolator
    
    # Create interpolator
    interp = RegularGridInterpolator((y, x), p, method='linear', bounds_error=False, fill_value=0)
    
    # Sample pressure on circle
    points = np.column_stack([y_circle, x_circle])
    p_circle = interp(points)
    
    # Integrate: F = -∫p n dS
    # Normal vectors point outward
    nx = np.cos(theta)
    ny = np.sin(theta)
    
    dtheta = 2 * np.pi / n_angles
    drag = -np.sum(p_circle * nx) * r_sample * dtheta  # x-force
    lift = -np.sum(p_circle * ny) * r_sample * dtheta  # y-force
    
    drag_forces.append(drag)
    lift_forces.append(lift)

lift_forces = np.array(lift_forces)
drag_forces = np.array(drag_forces)

# Plot
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].plot(t, drag_forces, 'b-', linewidth=1.5)
axes[0].set_xlabel('Time (s)')
axes[0].set_ylabel('Drag Force (approximate)')
axes[0].set_title('Drag Force vs Time')
axes[0].grid(True, alpha=0.3)

axes[1].plot(t, lift_forces, 'r-', linewidth=1.5)
axes[1].axhline(0, color='gray', linestyle='--', alpha=0.5)
axes[1].set_xlabel('Time (s)')
axes[1].set_ylabel('Lift Force (approximate)')
axes[1].set_title('Lift Force vs Time (Oscillation = Vortex Shedding)')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Mean drag: {drag_forces.mean():.4f}")
print(f"Lift oscillation amplitude: {(lift_forces.max() - lift_forces.min()) / 2:.4f}")

## Summary

This notebook demonstrated:

1. **Time-dependent simulation** of flow around a cylinder
2. **Vortex shedding** visualization (von Kármán street)
3. **Interactive time slider** to explore dynamics
4. **Space-time diagrams** showing oscillation patterns
5. **Force analysis** showing lift oscillation

### For Operator Learning

This dataset provides:
- **Input**: Inlet velocity (scalar) or initial state
- **Output**: Time trajectory (n_t, ny, nx, 3)
- **Challenge**: Predicting unsteady dynamics and vortex shedding frequency

Possible learning tasks:
1. **Autoregressive**: Predict next frame from current frame
2. **Full trajectory**: Map inlet conditions to entire evolution
3. **Frequency prediction**: Estimate shedding frequency from early transient